In [1]:
import json
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma

from dotenv.ipython import load_dotenv
import os

<h1>LLM</h1>

In [4]:
load_dotenv(override=True)

True

In [34]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)

<h1>Implementing RAG</h1>

<h2>1- Loading the PDF, Chunking</h2>

In [6]:
pdf_file="./pdfs/RapportStageAWB(Final).pdf"

In [7]:
pdf_loader = PyPDFLoader(pdf_file)

In [8]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path="./pdfs")    

In [9]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [11]:
chunks = loader.load_and_split(text_splitter)

In [12]:
len(chunks)

139

<h2>2- Vector Store- ChromaDB, Embeddings</h2>

In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


d:\Lab_DL\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2332.61it/s]


In [20]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="rapport_stage_awb",
    persist_directory="./store"
)

In [21]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [22]:
retrieved_chunks= retriever.invoke("Quels sont les objectifs du stage ?")

In [23]:
print(retrieved_chunks)

[Document(metadata={'author': 'Mohamed salem Berrada', 'creationdate': '2025-06-30T14:24:49+01:00', 'producer': 'Microsoft® Word pour Microsoft\xa0365', 'page_label': '13', 'moddate': '2025-06-30T14:24:49+01:00', 'total_pages': 86, 'page': 12, 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'source': 'pdfs\\RapportStageAWB(Final).pdf'}, page_content='3 \n \n \n  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nChapitre I : CADRAGE ET \nCONDUITE DU PROJET'), Document(metadata={'page': 8, 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'page_label': '9', 'total_pages': 86, 'author': 'Mohamed salem Berrada', 'moddate': '2025-06-30T14:24:49+01:00', 'producer': 'Microsoft® Word pour Microsoft\xa0365', 'creationdate': '2025-06-30T14:24:49+01:00', 'source': 'pdfs\\RapportStageAWB(Final).pdf'}, page_content='ix \n \n \n  \n \nTable des matières \nIntroduction Générale ........................................................................................

In [24]:
print(len(retrieved_chunks))

5


<h2>3- RAG Q&A </h2>

<h3>Prompt Design</h3>

In [38]:
prompt_template = """
Answer the following question based only on the provided context and in french
The context is about a intership report of a student at AWB company for churn prediction project.
The context is delimited by <context> tag
the user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS

<context>
  {context}
</context>
<question>
  {question}
</question>
"""

<h3>Retrieving the Relevant Documents </h3>

In [26]:
user_input = "Quels sont les objectifs du stage ?"

In [27]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [doc.page_content for doc in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [28]:
context_for_query

"3 \n \n \n  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nChapitre I : CADRAGE ET \nCONDUITE DU PROJET. ix \n \n \n  \n \nTable des matières \nIntroduction Générale ............................................................................................................. 1 \nChapitre I : CADRAGE ET CONDUITE DU PROJET ..................................................................... 3 \n1.1 Introduction ................................................................................................................... 4 \n1.2 Organisme d’accueil :............................................................................................... 4 \n1.2.1 Historique ............................................................................................................... 4 \n1.2.2 Services .................................................................................................................. 5 \n1.2.3 Organigramme de l’organisme d’accu

In [39]:
prompt=prompt_template.format(context=context_for_query, question=user_input)

In [40]:
print(prompt)


Answer the following question based only on the provided context and in french
The context is about a intership report of a student at AWB company for churn prediction project.
The context is delimited by <context> tag
the user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS

<context>
  3 
 
 
  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Chapitre I : CADRAGE ET 
CONDUITE DU PROJET. ix 
 
 
  
 
Table des matières 
Introduction Générale ............................................................................................................. 1 
Chapitre I : CADRAGE ET CONDUITE DU PROJET ..................................................................... 3 
1.1 Introduction ................................................................................................................... 4 
1.2 Organisme d’accueil :............................................................................................... 4 


In [41]:
resp = llm.invoke(prompt)

In [36]:
from IPython.display import Markdown

In [42]:
print(display(Markdown(resp.content)))

<think>
Okay, let's see. The user is asking about the objectives of the internship mentioned in the context. First, I need to look through the provided context to find where the objectives are discussed.

Looking at the table of contents, there's a section titled "1.3 Présentation du projet" which includes "1.3.3 Objectifs". That seems like the right place to check. However, when I look at the context provided, the actual content for chapter 1.3.3 isn't included here. The context provided cuts off after "1.3.4 Contraintes" and then goes into "1.6 Planification Prévisionnelle". 

The user's question is specifically about the objectives, which should be in section 1.3.3. But in the given context, there's no text under that section. The only parts available are the table of contents entries. Since the actual content of 1.3.3 isn't present in the context provided, I can't extract the objectives from here. 

Therefore, based on the information given, I have to conclude that the answer isn't found in the context. The user might need to refer to the full document for the details on the objectives.
</think>

JE NE SAIS PAS

None


<h3>Defining the RAG function for response</h3>

In [43]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp = llm.invoke(prompt)
    return resp.content

In [46]:
response = RAG("dans quelle entreprise le stage a-t-il eu lieu ?")
print(display(Markdown(response)))

<think>
Okay, let's see. The user is asking in French, "dans quelle entreprise le stage a-t-il eu lieu?" which translates to "In which company did the internship take place?" I need to find the answer in the provided context.

Looking through the context, there's a mention of "AWB company" in the initial description. Then, in the context text, under section 1.2.3, it says "Le service Pilotage de performance BDME, au sein de laquelle nous avons effectué notre stage de PFE, s’inscrit au sein du pôle Finances Groupe." The key part here is "au sein de laquelle nous avons effectué notre stage de PFE" which means "within which we carried out our internship." The organigramme (organigram) mentioned is labeled as "AWB Organigramme" in the context. 

So putting it together, the internship was at AWB company. The answer should be AWB. I need to check if there's any other possible name mentioned, but the context doesn't provide another company name. The answer is AWB.
</think>

Le stage a eu lieu dans l'entreprise AWB.

None


<h2>4- Evaluation</h2>

In [55]:
user_input = "Donne moi les metrics utilisées pour évaluer le modèle de churn prediction ?"

In [48]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [doc.page_content for doc in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [49]:
user_message_template = """
  ###Question
  {question}
  ###Context
  {context}
  ###Answer
  {answer}
"""

In [56]:
answer = RAG(user_input)
print(display(Markdown(answer)))

<think>
Okay, let's see. The user is asking for the metrics used to evaluate the churn prediction model. I need to look through the provided context to find any mentions of evaluation metrics.

Looking at the context, there's a section about evaluating performance and optimization. It mentions analyzing the impact of the decision threshold on main evaluation metrics: precision, recall, and F1-score. The text specifically says "précision, rappel et F1-score" in French. Also, there's a mention of using regression for predicting a probability score and modeling time before churn, but those are more about the model's approach rather than evaluation metrics. 

I don't see any other metrics listed besides precision, recall, and F1-score. The context doesn't mention accuracy, ROC-AUC, or others. The user wants the answer in French, so I should list those three metrics in French as they are mentioned. The answer should be concise and only include what's in the context. Since the question is in French, the answer should be in French too. Let me double-check to make sure there's no other metrics mentioned elsewhere in the context. Nope, just those three. So the answer should be those three metrics.
</think>

Les métriques utilisées pour évaluer le modèle de prédiction de l'attrition sont la **précision**, le **rappel** et le **F1-score**. Ces métriques ont été analysées en fonction du seuil de décision pour optimiser les performances du modèle.

None


<h3>Groundness </h3>

In [57]:
groundedness_rater_system_message = """
Vous etes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.

Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l'IA commencera par ###Answer.

Critères d'évaluation :
La tache consiste à juger dans quelle mesure la réponse respecte la métrique.

1 - La métrique n'est pas respectée du tout
2 - La métrique n'est respectée que dans une mesure limitée
3 - La métrique est respectée dans une mesure limitée
4 - La métrique est respectée en grande partie
5 - La métrique est entièrement respectée

Métrique d'évaluation :
La réponse doit être entièrement basée sur le contexte fourni.

Instructions :
Ecrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.

"""

In [58]:
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [59]:
def evaluate(system_message, user_message_template, question, model=groundness_checker):
    retrieved_chunks = retriever.invoke(question)
    context_list = [doc.page_content for doc in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
       {system_message}\n
       USER:
       {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response = model.invoke(prompt)
    return juge_response.content

In [60]:
resp = evaluate(groundedness_rater_system_message, user_message_template, user_input)

In [61]:
print(display(Markdown(resp)))

**Étapes pour évaluer la conformité de la réponse à la métrique « la réponse doit être entièrement basée sur le contexte fourni »**

1. **Identifier la question** – L’utilisateur demande : « Donne moi les metrics utilisées pour évaluer le modèle de churn prediction ? »

2. **Extraire du contexte toutes les mentions de métriques d’évaluation** –  
   - Le texte indique : « nous avons analysé l’impact du seuil de décision sur les principales métriques d’évaluation : précision, rappel et F1‑score. »  
   - Aucun autre indicateur (ex. : exactitude, ROC‑AUC, etc.) n’est mentionné dans le contexte.

3. **Comparer le contenu de la réponse générée avec les éléments extraits** –  
   - La réponse cite exactement les trois métriques : précision, rappel et F1‑score.  
   - Elle ajoute une phrase explicative (« Ces métriques ont été analysées pour déterminer le seuil de décision optimal du modèle ») qui reprend le même passage du contexte (analyse de l’impact du seuil de décision).

4. **Vérifier l’absence d’informations extérieures ou d’interprétations non présentes dans le contexte** –  
   - La réponse ne mentionne aucune métrique supplémentaire, aucune définition, ni aucun chiffre qui ne figure pas dans le texte fourni.  
   - Le style et le vocabulaire (français, termes « précision, rappel, F1‑score ») sont cohérents avec le contexte.

5. **Attribuer la note** – Si la réponse reproduit fidèlement les informations du contexte sans ajouter d’éléments externes, la métrique est entièrement respectée → note **5**.

---

**Évaluation de la réponse fournie**

- **Conformité au contexte** : La réponse se limite aux seules métriques explicitement listées dans le texte (précision, rappel, F1‑score) et reprend la justification liée à l’analyse du seuil de décision, exactement comme le décrit le contexte.
- **Absence d’informations hors‑contexte** : Aucun élément supplémentaire n’est introduit.
- **Exactitude** : Les métriques mentionnées correspondent parfaitement à celles du passage « principales métriques d’évaluation ».

**Conclusion** : La réponse respecte intégralement la métrique demandée.

**Note finale : 5**.

None
